# CIP alerting system

This notebook compares actual CIP cycles from the log files with the object-specific thresholds in `CIP Pre Rinse Timer details.xlsx`.

A cycle is defined as a continuous block of records for one object. A gap of more than 30 minutes starts the next cycle. The log's `Time In Seconds` is cumulative within a cycle, so each activity duration is calculated as its ending counter minus its starting counter.


In [9]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('datasets')
LOG_FILES = sorted(DATA_DIR.glob('CIP_Log*.xlsx'))
REFERENCE_FILE = DATA_DIR / 'CIP Pre Rinse Timer details.xlsx'

# These are deliberately visible and editable.
CYCLE_GAP_MINUTES = 30
DURATION_TOLERANCE = 0.10  # Alert when actual duration exceeds threshold + 10%
FLOW_TOLERANCE = 0.10      # 10% below the reference flow target

print(f'{len(LOG_FILES)} log files found')
print(f'Reference workbook: {REFERENCE_FILE.name}')

if not LOG_FILES:
    raise FileNotFoundError(f'No CIP log files found in {DATA_DIR.resolve()}')
if not REFERENCE_FILE.exists():
    raise FileNotFoundError(f'Reference workbook not found: {REFERENCE_FILE.resolve()}')


4 log files found
Reference workbook: CIP Pre Rinse Timer details.xlsx


In [10]:
REQUIRED = [
    'Date Time', 'Object', 'Recipe', 'Activity', 'Step',
    'Time In Seconds', 'Temp SP', 'Return Temp', 'Supply Flow',
    'Return Flow', 'Min Return Flow SP', 'Return Conducivity',
    'Caustic SP', 'Acid SP', 'Final Rinse Cond. SP', 'Flip'
]

def line_from_report(path):
    raw = pd.read_excel(path, header=None, nrows=10)
    text = ' '.join(raw.astype(str).fillna('').stack())
    match = re.search(r'Line\s*:\s*([^ ]+)', text, flags=re.I)
    return match.group(1) if match else path.stem

def read_log(path):
    df = pd.read_excel(path, sheet_name=0, header=13)
    df = df[REQUIRED].copy()
    df['Date Time'] = pd.to_datetime(df['Date Time'], errors='coerce')
    df = df[df['Date Time'].notna()].copy()
    for col in REQUIRED[5:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['Line'] = line_from_report(path)
    df['Source File'] = path.name
    return df

logs = pd.concat([read_log(path) for path in LOG_FILES], ignore_index=True)
logs = logs.sort_values(['Line', 'Object', 'Date Time']).reset_index(drop=True)
print(f'{len(logs):,} timestamped log rows loaded')
logs[['Line', 'Object', 'Activity', 'Step']].head()


38,943 timestamped log rows loaded


,Line,Object,Activity,Step
0,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill To Tank
1,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill To Tank
2,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill Circuit
3,M01W01,C01C01-RMT Tank-1,Pre Rinse,Circulation
4,M01W01,C01C01-RMT Tank-1,Pre Rinse,Circulation


In [11]:
summary = pd.read_excel(REFERENCE_FILE, sheet_name='Summary')
process = pd.read_excel(REFERENCE_FILE, sheet_name='Process CIP Water Consumption')

def object_key(value):
    # Uses the equipment code shared by the reference and log names, e.g. C02C01.
    match = re.search(r'C\d{2}[A-Z]\d{2}', str(value).upper())
    return match.group(0) if match else str(value).strip().upper()

summary['Object Key'] = summary['Object'].map(object_key)
process['Object Key'] = process['Object'].map(object_key)
logs['Object Key'] = logs['Object'].map(object_key)

final_rinse_seconds = [
    'Final Rinse Fill to Tank (sec)',
    'Final Rinse Fill Circuit (Sec)',
    'Final Rinse Front Mix Zone (Sec)',
    'Final Rinse Circulation (Sec)',
      ]

thresholds = summary.merge(
    process[['Object', 'Suppy Flow (L/Hr)', 'Return Flow (L/Hr)', *final_rinse_seconds]],
    on='Object',
    how='left',
    validate='one_to_one'
)
thresholds['Final Rinse Total Time (min)'] = (
    thresholds[final_rinse_seconds].sum(axis=1, min_count=len(final_rinse_seconds)) / 60
)
thresholds = thresholds.rename(columns={
    'New Proposed Pre Rinse total time(Min)': 'Pre Rinse Threshold (min)',
    'Caustic- Total Time': 'Caustic Threshold (min)',
    'Acid Total Time': 'Acid Threshold (min)',
    'Suppy Flow (L/Hr)': 'Supply Flow Threshold (L/hr)',
    'Return Flow (L/Hr)': 'Return Flow Threshold (L/hr)'
})

threshold_view = thresholds[[
    'Object', 'Object Key', 'Pre Rinse Threshold (min)',
    'Caustic Threshold (min)', 'Acid Threshold (min)',
    'Final Rinse Total Time (min)', 'Supply Flow Threshold (L/hr)',
    'Return Flow Threshold (L/hr)'
]].copy()
threshold_view.head()
reference_objects = thresholds[['Object', 'Object Key']].copy()

def reference_object_for_log(log_object):
    key = object_key(log_object)
    candidates = reference_objects[reference_objects['Object Key'].eq(key)]
    if candidates.empty:
        return np.nan
    if len(candidates) == 1:
        return candidates.iloc[0]['Object']
    log_text = str(log_object).upper()
    scores = candidates['Object'].map(
        lambda value: sum(token in log_text for token in re.findall(r'[A-Z0-9]+', str(value).upper()))
    )
    return candidates.loc[scores.idxmax(), 'Object']

logs['Reference Object'] = logs['Object'].map(reference_object_for_log)
print(f"{logs['Reference Object'].notna().sum():,} log rows matched to reference objects")


20,665 log rows matched to reference objects


### Final-rinse calculation

The final-rinse total is calculated as:

`Fill to Tank + Fill Circuit + Front Mix Zone + Circulation`

All four reference fields are in seconds and represent sequential final-rinse steps, so adding them and dividing by 60 is the correct first calculation. If the PLC runs any of these steps in parallel, that assumption should be changed after confirming it with the process owner.


In [12]:
logs = logs.sort_values(['Line', 'Object', 'Date Time']).copy()
new_cycle = (logs.groupby(['Line', 'Object'])['Date Time']
             .diff().gt(pd.Timedelta(minutes=CYCLE_GAP_MINUTES)))
logs['Cycle Number'] = (new_cycle.fillna(True)
                       .groupby([logs['Line'], logs['Object']])
                       .cumsum().astype(int))

cycle_keys = ['Line', 'Object', 'Reference Object', 'Object Key', 'Cycle Number']
cycle_info = (logs.groupby(cycle_keys, as_index=False, dropna=False)
              .agg({'Date Time': ['min', 'max', 'size'],
                    'Recipe': lambda x: ', '.join(sorted(set(x.dropna().astype(str))))}))
cycle_info.columns = cycle_keys + ['Cycle Start', 'Cycle End', 'Record Count', 'Recipes']

activity_times = (logs.groupby(cycle_keys + ['Activity'], as_index=False, dropna=False)
                  .agg({'Time In Seconds': ['min', 'max']}))
activity_times.columns = cycle_keys + ['Activity', 'Start Seconds', 'End Seconds']
activity_times['Activity Time (min)'] = (
    activity_times['End Seconds'] - activity_times['Start Seconds']
) / 60
activity_times.loc[activity_times['Activity Time (min)'] < 0, 'Activity Time (min)'] = np.nan

activity_times = (activity_times
                  .pivot(index=cycle_keys, columns='Activity', values='Activity Time (min)')
                  .reset_index()
                  .rename_axis(None, axis=1))

flow_times = (logs.groupby(cycle_keys, as_index=False, dropna=False)
              .agg({'Supply Flow': 'median', 'Return Flow': ['median', 'min']}))
flow_times.columns = cycle_keys + ['Supply Flow Actual (L/hr)', 'Return Flow Actual (L/hr)', 'Minimum Return Flow Seen (L/hr)']

cycle_results = (cycle_info.merge(activity_times, on=cycle_keys, how='left')
                .merge(flow_times, on=cycle_keys, how='left'))
cycle_results.head()


,Line,Object,Reference Object,Object Key,Cycle Number,Cycle Start,Cycle End,Record Count,Recipes,Acid,Caustic,Final Drain,Final Rinse,Mid Rinse,Pre Rinse,Supply Flow Actual (L/hr),Return Flow Actual (L/hr),Minimum Return Flow Seen (L/hr)
0,M01W01,C01C01-RMT Tank-1,NaN,C01C01,0,2026-07-02 19:38:06.703,2026-07-02 20:14:06.703,37,Caustic,NaN,23.000000,0.0,5.000000,NaN,5.000000,12784.38,20220.0,-1045.0
1,M01W01,C01C01-RMT Tank-1,NaN,C01C01,1,2026-07-04 00:43:06.703,2026-07-04 01:36:06.653,54,Caustic,NaN,38.016667,0.0,6.983333,NaN,5.000000,11404.69,267.5,-50.0
2,M01W01,C01C01-RMT Tank-1,NaN,C01C01,2,2026-07-04 19:54:06.703,2026-07-04 21:14:06.653,81,Caustic Acid,12.0,37.000000,NaN,10.000000,9.0,7.983333,12403.13,3115.0,-1045.0
3,M01W01,C01C01-RMT Tank-1,NaN,C01C01,3,2026-07-06 07:06:06.703,2026-07-06 08:14:06.703,69,Caustic,NaN,51.983333,NaN,10.016667,NaN,4.000000,862.50,-45.0,-1045.0
4,M01W01,C01C01-RMT Tank-1,NaN,C01C01,4,2026-07-07 06:19:06.703,2026-07-07 07:19:06.703,61,Caustic,NaN,46.000000,NaN,7.000000,NaN,5.000000,281.25,-45.0,-560.0


In [13]:
actual_to_threshold = {
    'Pre Rinse': ('Pre Rinse', 'Pre Rinse Threshold (min)'),
    'Caustic': ('Caustic', 'Caustic Threshold (min)'),
    'Acid': ('Acid', 'Acid Threshold (min)'),
    'Final Rinse': ('Final Rinse', 'Final Rinse Total Time (min)')
}

threshold_lookup = thresholds.set_index('Object')

for phase, (actual_col, threshold_col) in actual_to_threshold.items():
    cycle_results[f'{phase} Threshold (min)'] = cycle_results['Reference Object'].map(
        threshold_lookup[threshold_col])
    cycle_results[f'{phase} Breach'] = (
        cycle_results[actual_col].notna()
        & cycle_results[f'{phase} Threshold (min)'].notna()
        & (cycle_results[actual_col] > cycle_results[f'{phase} Threshold (min)'] * (1 + DURATION_TOLERANCE))
    )

cycle_results['Supply Flow Threshold (L/hr)'] = cycle_results['Reference Object'].map(
    threshold_lookup['Supply Flow Threshold (L/hr)'])
cycle_results['Return Flow Threshold (L/hr)'] = cycle_results['Reference Object'].map(
    threshold_lookup['Return Flow Threshold (L/hr)'])
cycle_results['Supply Flow Breach'] = (
    cycle_results['Supply Flow Actual (L/hr)'].notna()
    & cycle_results['Supply Flow Threshold (L/hr)'].notna()
    & (cycle_results['Supply Flow Actual (L/hr)'] < cycle_results['Supply Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
)
cycle_results['Return Flow Breach'] = (
    cycle_results['Return Flow Actual (L/hr)'].notna()
    & cycle_results['Return Flow Threshold (L/hr)'].notna()
    & (cycle_results['Return Flow Actual (L/hr)'] < cycle_results['Return Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
)

breach_cols = [c for c in cycle_results.columns if c.endswith('Breach')]
cycle_results['Alert Count'] = cycle_results[breach_cols].sum(axis=1)
cycle_results['Status'] = np.select(
    [cycle_results['Reference Object'].isna(), cycle_results['Alert Count'] > 0],
    ['NO REFERENCE', 'ALERT'],
    default='OK'
)

cycle_summary_columns = cycle_keys + ['Cycle Start', 'Cycle End', 'Recipes',
    'Pre Rinse', 'Caustic', 'Acid', 'Final Rinse',
    'Supply Flow Actual (L/hr)', 'Return Flow Actual (L/hr)',
    'Alert Count', 'Status']
cycle_results[cycle_summary_columns].head(20)


,Line,Object,Reference Object,Object Key,Cycle Number,Cycle Start,Cycle End,Recipes,Pre Rinse,Caustic,Acid,Final Rinse,Supply Flow Actual (L/hr),Return Flow Actual (L/hr),Alert Count,Status
0,M01W01,C01C01-RMT Tank-1,NaN,C01C01,0,2026-07-02 19:38:06.703,2026-07-02 20:14:06.703,Caustic,5.000000,23.000000,NaN,5.000000,12784.3800,20220.0,0,NO REFERENCE
1,M01W01,C01C01-RMT Tank-1,NaN,C01C01,1,2026-07-04 00:43:06.703,2026-07-04 01:36:06.653,Caustic,5.000000,38.016667,NaN,6.983333,11404.6900,267.5,0,NO REFERENCE
2,M01W01,C01C01-RMT Tank-1,NaN,C01C01,2,2026-07-04 19:54:06.703,2026-07-04 21:14:06.653,Caustic Acid,7.983333,37.000000,12.000000,10.000000,12403.1300,3115.0,0,NO REFERENCE
3,M01W01,C01C01-RMT Tank-1,NaN,C01C01,3,2026-07-06 07:06:06.703,2026-07-06 08:14:06.703,Caustic,4.000000,51.983333,NaN,10.016667,862.5000,-45.0,0,NO REFERENCE
4,M01W01,C01C01-RMT Tank-1,NaN,C01C01,4,2026-07-07 06:19:06.703,2026-07-07 07:19:06.703,Caustic,5.000000,46.000000,NaN,7.000000,281.2500,-45.0,0,NO REFERENCE
5,M01W01,C01C01-RMT Tank-1,NaN,C01C01,5,2026-07-08 07:32:06.703,2026-07-08 08:10:06.653,Caustic,7.000000,23.000000,NaN,6.000000,12643.7500,11405.0,0,NO REFERENCE
6,M01W01,C01C01-RMT Tank-1,NaN,C01C01,6,2026-07-09 14:57:06.703,2026-07-09 15:40:06.703,Caustic,5.983333,29.016667,NaN,5.000000,12745.3100,18422.5,0,NO REFERENCE
7,M01W01,C01C01-RMT Tank-1,NaN,C01C01,7,2026-07-10 21:49:06.653,2026-07-10 22:52:06.703,"Caustic, Caustic Acid",5.016667,25.016667,11.000000,7.016667,12748.4400,15887.5,0,NO REFERENCE
8,M01W01,C01C01-RMT Tank-1,NaN,C01C01,8,2026-07-12 09:04:06.653,2026-07-12 09:48:06.703,Caustic,4.000000,32.016667,NaN,6.000000,11703.1300,-40.0,0,NO REFERENCE
9,M01W01,C01C01-RMT Tank-1,NaN,C01C01,9,2026-07-12 17:47:06.703,2026-07-12 17:55:06.653,"Caustic, Final rinse",0.000000,NaN,NaN,5.000000,8087.5000,-40.0,0,NO REFERENCE


In [14]:
alert_rows = []
for _, row in cycle_results.iterrows():
    base = row[cycle_keys + ['Cycle Start', 'Cycle End']].to_dict()
    for phase, (actual_col, threshold_col) in actual_to_threshold.items():
        if bool(row[f'{phase} Breach']):
            alert_rows.append({**base, 'Alert Type': f'{phase} duration',
                               'Actual Value': row[actual_col], 'Threshold': row[f'{phase} Threshold (min)'],
                               'Unit': 'minutes'})
    for signal, actual_col, threshold_col in [
        ('Supply flow', 'Supply Flow Actual (L/hr)', 'Supply Flow Threshold (L/hr)'),
        ('Return flow', 'Return Flow Actual (L/hr)', 'Return Flow Threshold (L/hr)')
    ]:
        if bool(row[f'{signal.title()} Breach']):
            alert_rows.append({**base, 'Alert Type': signal,
                               'Actual Value': row[actual_col], 'Threshold': row[threshold_col],
                               'Unit': 'L/hr'})

alerts = pd.DataFrame(alert_rows)
if alerts.empty:
    alerts = pd.DataFrame(columns=cycle_keys + ['Cycle Start', 'Cycle End', 'Alert Type', 'Actual Value', 'Threshold', 'Unit'])

print(f'{len(cycle_results):,} cycles identified')
print(f'{len(alerts):,} threshold breaches found')
alerts.head(50)


699 cycles identified
304 threshold breaches found


,Line,Object,Reference Object,Object Key,Cycle Number,Cycle Start,Cycle End,Alert Type,Actual Value,Threshold,Unit
0,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,1,2026-07-02 07:54:06.653,2026-07-02 08:53:06.653,Pre Rinse duration,15.983333,14.500000,minutes
1,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,5,2026-07-05 00:03:06.653,2026-07-05 03:17:06.703,Acid duration,115.783333,24.500000,minutes
2,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,13,2026-07-09 18:48:06.703,2026-07-09 20:37:06.703,Caustic duration,90.000000,33.500000,minutes
3,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,14,2026-07-10 09:01:06.653,2026-07-10 10:25:06.703,Pre Rinse duration,31.000000,14.500000,minutes
4,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,14,2026-07-10 09:01:06.653,2026-07-10 10:25:06.703,Caustic duration,42.000000,33.500000,minutes
5,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,14,2026-07-10 09:01:06.653,2026-07-10 10:25:06.703,Return flow,6637.500000,7425.000000,L/hr
6,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,17,2026-07-12 09:45:06.703,2026-07-12 10:44:06.653,Caustic duration,38.016667,33.500000,minutes
7,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,20,2026-07-13 20:53:06.703,2026-07-13 23:30:06.653,Caustic duration,88.983333,33.500000,minutes
8,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,20,2026-07-13 20:53:06.703,2026-07-13 23:30:06.653,Acid duration,31.983333,24.500000,minutes
9,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,21,2026-07-14 07:58:06.703,2026-07-14 10:08:06.703,Caustic duration,111.000000,33.500000,minutes


In [15]:
unmatched = (cycle_results.loc[~cycle_results['Reference Object'].isin(threshold_lookup.index),
                             ['Line', 'Object', 'Object Key']]
             .drop_duplicates()
             .sort_values(['Line', 'Object']))

cycle_counts = (cycle_results.groupby(['Line', 'Object'], as_index=False)
                .agg(Cycles=('Cycle Number', 'nunique'))
                .sort_values(['Line', 'Object']))
print('Cycles per object:')
cycle_counts.head(50)

print('Objects without a matching reference threshold:')
unmatched


Cycles per object:
Objects without a matching reference threshold:


,Line,Object,Object Key
0,M01W01,C01C01-RMT Tank-1,C01C01
21,M01W01,C01C02-RMT Tank-2,C01C02
44,M01W01,C01C03-RMT Tank-3,C01C03
71,M01W01,C01W03-RMT To Reception line,C01W03
92,M01W01,R01W01-Milk Reception Line,R01W01-MILK RECEPTION LINE
116,M01W02,M01W02_CC01 - Raw Milk Tanker,M01W02_CC01 - RAW MILK TANKER
421,M11W02,C01W02-Milk Past-2,C01W02


## How to use this notebook

Run the cells from top to bottom. The main outputs are `cycle_results`, which contains one row per object cycle, and `alerts`, which contains one row per breach.

The current flow comparison uses the median flow observed during a cycle. That avoids triggering an alert from one bad sensor reading. The duration and flow tolerances are working defaults, not confirmed plant rules. Confirm them with the CIP owner before treating the alerts as operational alarms.


## Daily alert summary

The next cell shows total cycles, alert events, and cycles with alerts for each calendar day.


In [ ]:
cycle_days = cycle_results.assign(**{'Alert Date': pd.to_datetime(cycle_results['Cycle Start']).dt.date})
daily_cycles = (cycle_days.groupby('Alert Date').size().rename('Cycles').to_frame())

alert_days = alerts.assign(**{'Alert Date': pd.to_datetime(alerts['Cycle Start']).dt.date})
daily_alerts = (alert_days.groupby('Alert Date')
                .agg(**{'Alert Count': ('Alert Type', 'size'),
                       'Cycles with Alerts': ('Cycle Number', 'nunique')}))

alerts_per_day = (daily_cycles.join(daily_alerts, how='left')
                  .fillna({'Alert Count': 0, 'Cycles with Alerts': 0})
                  .reset_index())
alerts_per_day['Alert Count'] = alerts_per_day['Alert Count'].astype(int)
alerts_per_day['Cycles with Alerts'] = alerts_per_day['Cycles with Alerts'].astype(int)
alerts_per_day.sort_values('Alert Date')


### Check: Acid threshold vs Acid actual

The Summary sheet's `Acid Total Time` is **Mid Rinse + Acid Circulation** (from the Process sheet), but cell 6 compares it with the log's `Acid` activity only. The cell below proves the composition and shows how many acid alerts change when Mid Rinse is included in the actual.

In [17]:
# Diagnostic: what 'Acid Total Time' in the Summary sheet is made of,
# and what the notebook compares it against.
mid_rinse_seconds = [
    'Mid Rinse Fill to Tank (Sec)',
    'Mid Rinse Fill Circuit (Sec)',
    'Mid Rinse Front Mix Zone (Sec)',
    'Mid rinse Circulation (sec)',
]

acid_check = summary[['Object', 'Acid Total Time']].merge(
    process[['Object', *mid_rinse_seconds, 'Acid Circulation']], on='Object', how='left')
acid_check['Mid Rinse (min)'] = acid_check[mid_rinse_seconds].sum(axis=1) / 60
acid_check['Acid Circulation (min)'] = acid_check['Acid Circulation'] / 60
acid_check['Mid Rinse + Acid (min)'] = acid_check['Mid Rinse (min)'] + acid_check['Acid Circulation (min)']
acid_check['Matches Summary'] = np.isclose(acid_check['Mid Rinse + Acid (min)'], acid_check['Acid Total Time'])

print(f"Summary 'Acid Total Time' = Mid Rinse + Acid Circulation for "
      f"{acid_check['Matches Summary'].sum()} of {len(acid_check)} objects")
display(acid_check[['Object', 'Mid Rinse (min)', 'Acid Circulation (min)',
                    'Mid Rinse + Acid (min)', 'Acid Total Time', 'Matches Summary']].round(2))

# The notebook compares that combined threshold against the log's 'Acid' activity only.
acid_cycles = cycle_results[cycle_results['Acid'].notna() & cycle_results['Acid Threshold (min)'].notna()].copy()
acid_cycles['Mid Rinse + Acid Actual (min)'] = acid_cycles[['Mid Rinse', 'Acid']].sum(axis=1)
acid_cycles['Breach (Mid Rinse + Acid)'] = (
    acid_cycles['Mid Rinse + Acid Actual (min)'] > acid_cycles['Acid Threshold (min)'] * (1 + DURATION_TOLERANCE))

print(f"Acid cycles checked: {len(acid_cycles)}")
print(f"Acid alerts now (Acid only vs combined threshold): {acid_cycles['Acid Breach'].sum()}")
print(f"Acid alerts when actual also includes Mid Rinse: {acid_cycles['Breach (Mid Rinse + Acid)'].sum()}")
acid_cycles[['Object', 'Cycle Number', 'Cycle Start', 'Mid Rinse', 'Acid',
             'Mid Rinse + Acid Actual (min)', 'Acid Threshold (min)',
             'Acid Breach', 'Breach (Mid Rinse + Acid)']].round(2)

Summary 'Acid Total Time' = Mid Rinse + Acid Circulation for 35 of 35 objects


,Object,Mid Rinse (min),Acid Circulation (min),Mid Rinse + Acid (min),Acid Total Time,Matches Summary
0,M11W01_03_C02C01_PMT-1,10.67,7.0,17.67,17.67,True
1,M11W01_04_C02C02_PMT-2,10.67,7.0,17.67,17.67,True
2,M11W01_05_C02C03_PMT-3,10.67,7.0,17.67,17.67,True
3,M11W01_06_C02C04_PMT-4,10.67,7.0,17.67,17.67,True
4,M11W01_07_C02W01_PMT-RECIRCULATION,8.02,3.0,11.02,11.02,True
5,M11W01_08_C02W02_PMT TO HSM,11.50,3.0,14.50,14.50,True
6,M11W01_09_C02W03_PMT TO CPT,11.03,3.0,14.03,14.03,True
7,M11W01_10_C02W04_PMT TO MILK PAST,11.17,3.0,14.17,14.17,True
8,M11W01_11_C02W05_PMT TO PCT,9.87,3.0,12.87,12.87,True
9,M11W01_12_C01W01_MILK PAST-1,17.50,7.0,24.50,24.50,True


Acid cycles checked: 61
Acid alerts now (Acid only vs combined threshold): 17
Acid alerts when actual also includes Mid Rinse: 31


,Object,Cycle Number,Cycle Start,Mid Rinse,Acid,Mid Rinse + Acid Actual (min),Acid Threshold (min),Acid Breach,Breach (Mid Rinse + Acid)
173,C01W01-Milk Past-1,5,2026-07-05 00:03:06.653,9.02,115.78,124.80,24.50,True,True
176,C01W01-Milk Past-1,8,2026-07-06 20:00:06.653,9.02,16.00,25.02,24.50,False,False
188,C01W01-Milk Past-1,20,2026-07-13 20:53:06.703,8.00,31.98,39.98,24.50,True,True
200,C01W01-Milk Past-1,32,2026-07-21 00:28:06.653,9.98,16.00,25.98,24.50,False,False
211,C02C01-PMT TANK-1,2,2026-07-04 03:50:06.653,10.02,16.00,26.02,17.67,False,True
...,...,...,...,...,...,...,...,...,...
673,C11W01-CPT To EVA BTD-2,24,2026-07-26 04:55:06.653,6.00,4.00,10.00,12.52,False,False
676,C11W03-CPT To EVA BTD-1,2,2026-07-05 00:25:06.703,13.00,6.00,19.00,12.53,False,True
677,C11W03-CPT To EVA BTD-1,3,2026-07-05 02:33:06.703,6.98,5.02,12.00,12.53,False,False
692,C11W03-CPT To EVA BTD-1,18,2026-07-19 23:42:06.703,7.00,5.00,12.00,12.53,False,False


### Check: one "cycle" containing several CIP runs

Cycles are split only on a time gap of more than `CYCLE_GAP_MINUTES`. `Time In Seconds` restarts from a small value at the start of each CIP run, so if the counter drops inside a cycle, two or more runs with no gap between them were merged into one cycle. The activity times for those cycles are then measured from the first run's start to the last run's end.

In [18]:
# Check: cycles where the elapsed counter drops (a new CIP run started inside the cycle)
counter_drop = logs.groupby(cycle_keys, dropna=False)['Time In Seconds'].diff().lt(0)
merged_cycles = (logs[counter_drop]
                 .groupby(cycle_keys, dropna=False).size()
                 .rename('Counter Resets').reset_index()
                 .merge(cycle_results[cycle_keys + ['Cycle Start', 'Cycle End', 'Record Count',
                                                    'Status', 'Pre Rinse', 'Caustic', 'Final Rinse']],
                        on=cycle_keys))
merged_cycles['Cycle Hours'] = (merged_cycles['Cycle End'] - merged_cycles['Cycle Start']).dt.total_seconds() / 3600

print(f"{len(merged_cycles)} of {len(cycle_results)} cycles contain more than one CIP run "
      f"({merged_cycles['Counter Resets'].sum()} counter resets)")
display(merged_cycles.groupby(['Line', 'Object'])
        .agg(Merged_Cycles=('Cycle Number', 'size'), Counter_Resets=('Counter Resets', 'sum')))

# Example: the records around a reset in a matched object
example = merged_cycles[merged_cycles['Status'].ne('NO REFERENCE')].iloc[0]
example_rows = logs[(logs['Object'] == example['Object']) & (logs['Cycle Number'] == example['Cycle Number'])]
reset_at = example_rows.index[counter_drop.loc[example_rows.index]][0]
position = example_rows.index.get_loc(reset_at)
print(f"\nExample: {example['Object']} cycle {example['Cycle Number']} - counter drops at row {reset_at}")
display(example_rows.iloc[max(position - 3, 0):position + 3][['Date Time', 'Activity', 'Step', 'Time In Seconds']])
display(merged_cycles.sort_values('Cycle Hours', ascending=False)
        [['Object', 'Cycle Number', 'Cycle Start', 'Cycle Hours', 'Record Count',
          'Counter Resets', 'Pre Rinse', 'Caustic', 'Final Rinse', 'Status']].head(10).round(2))

53 of 699 cycles contain more than one CIP run (187 counter resets)


Merged_Cycles  Counter_Resets
Line   Object                                                      
M01W01 C01C01-RMT Tank-1                          1               1
       R01W01-Milk Reception Line                 1               1
M01W02 M01W02_CC01 - Raw Milk Tanker             41             174
M11W01 C02C01-PMT TANK-1                          2               2
       C02C02-PMT TANK-2                          2               2
       C02C03-PMT TANK-3                          2               2
       C02W02-PMT To HSM Line                     1               2
M11W02 C04C01-Mixing Tank-1                       1               1
       C04W02-Mixing Tank To CPT                  1               1
       C11C02-CPT Tank-2                          1               1


Example: C02C01-PMT TANK-1 cycle 3 - counter drops at row 18906


,Date Time,Activity,Step,Time In Seconds
18903,2026-07-06 00:01:06.653,Pre Rinse,Circulation,1756
18904,2026-07-06 00:02:06.703,Pre Rinse,Circulation,1817
18905,2026-07-06 00:03:06.653,Pre Rinse,Circulation,1876
18906,2026-07-06 00:06:06.653,Pre Rinse,Fill To Tank,25
18907,2026-07-06 00:14:06.703,Pre Rinse,Fill To Tank,52
18908,2026-07-06 00:15:06.703,Pre Rinse,Empty Tank,112


,Object,Cycle Number,Cycle Start,Cycle Hours,Record Count,Counter Resets,Pre Rinse,Caustic,Final Rinse,Status
5,M01W02_CC01 - Raw Milk Tanker,4,2026-07-04 22:52:06.703,9.20,442,8,9.08,40.62,20.85,NO REFERENCE
13,M01W02_CC01 - Raw Milk Tanker,15,2026-07-09 22:50:06.703,9.13,424,8,10.22,41.00,22.35,NO REFERENCE
22,M01W02_CC01 - Raw Milk Tanker,24,2026-07-14 23:44:06.653,8.73,426,8,7.83,46.92,30.08,NO REFERENCE
12,M01W02_CC01 - Raw Milk Tanker,14,2026-07-08 21:37:06.653,8.70,403,9,7.45,33.98,17.18,NO REFERENCE
19,M01W02_CC01 - Raw Milk Tanker,21,2026-07-12 21:13:06.703,8.63,414,8,9.02,35.73,19.57,NO REFERENCE
41,M01W02_CC01 - Raw Milk Tanker,50,2026-07-24 21:46:06.703,8.48,411,8,12.80,36.70,41.73,NO REFERENCE
4,M01W02_CC01 - Raw Milk Tanker,3,2026-07-03 22:28:06.653,8.47,397,8,8.72,35.22,20.98,NO REFERENCE
3,M01W02_CC01 - Raw Milk Tanker,1,2026-07-02 21:31:06.703,8.23,418,7,7.78,103.20,88.68,NO REFERENCE
2,M01W02_CC01 - Raw Milk Tanker,0,2026-07-01 22:27:06.703,7.87,393,8,7.50,29.40,13.38,NO REFERENCE
32,M01W02_CC01 - Raw Milk Tanker,39,2026-07-19 21:36:06.653,7.55,377,8,8.02,27.48,30.43,NO REFERENCE


### Check: flow alerts based on whole-cycle medians

Cell 5 takes the median supply and return flow over **every** record of the cycle: fill, drain, idle and the negative readings logged when the pump is off (about −50 L/hr). The flow targets apply to circulation. The cell below recalculates the median using only positive readings from `Circulation` steps and checks which flow alerts still breach.

In [19]:
# Check: how many flow alerts survive when only positive circulation readings are used
flow_columns = {'Supply flow': 'Supply Flow', 'Return flow': 'Return Flow'}
for signal, column in flow_columns.items():
    share = logs[column].lt(0).mean()
    print(f"{column}: {logs[column].lt(0).sum():,} negative readings ({share:.1%} of all records)")

circulation = logs[logs['Step'].eq('Circulation')]
circulation_median = {
    signal: circulation[circulation[column] > 0].groupby(cycle_keys)[column].median()
    for signal, column in flow_columns.items()
}

flow_alerts = alerts[alerts['Alert Type'].isin(flow_columns)].copy()
flow_alerts['Circulation Median (L/hr)'] = [
    circulation_median[row['Alert Type']].get(tuple(row[key] for key in cycle_keys), np.nan)
    for _, row in flow_alerts.iterrows()
]
flow_alerts['Still Breach'] = flow_alerts['Circulation Median (L/hr)'] < flow_alerts['Threshold'] * (1 - FLOW_TOLERANCE)

print()
display(flow_alerts.groupby('Alert Type')
        .agg(Alerts_Now=('Still Breach', 'size'),
             Median_At_Or_Below_Zero=('Actual Value', lambda s: s.le(0).sum()),
             Still_Breach_In_Circulation=('Still Breach', 'sum')))
flow_alerts[['Object', 'Cycle Number', 'Alert Type', 'Actual Value',
             'Circulation Median (L/hr)', 'Threshold', 'Still Breach']].round(0)

Supply Flow: 6,433 negative readings (16.5% of all records)
Return Flow: 6,027 negative readings (15.5% of all records)



,Alerts_Now,Median_At_Or_Below_Zero,Still_Breach_In_Circulation
Alert Type,,,
Return flow,57,9,18
Supply flow,26,8,0


,Object,Cycle Number,Alert Type,Actual Value,Circulation Median (L/hr),Threshold,Still Breach
5,C01W01-Milk Past-1,14,Return flow,6638.0,6941.0,7425.0,False
13,C01W01-Milk Past-1,26,Supply flow,-88.0,7494.0,7500.0,False
14,C01W01-Milk Past-1,26,Return flow,-109.0,7558.0,7425.0,False
20,C02C01-PMT TANK-1,3,Supply flow,-88.0,13997.0,14000.0,False
21,C02C01-PMT TANK-1,3,Return flow,-109.0,17433.0,12600.0,False
...,...,...,...,...,...,...,...
289,C11C02-CPT Tank-2,17,Return flow,609.0,15495.0,12600.0,False
294,C11C02-CPT Tank-2,21,Return flow,7477.0,16047.0,12600.0,False
297,C11C02-CPT Tank-2,24,Return flow,10753.0,16327.0,12600.0,False
300,C11W01-CPT To EVA BTD-2,2,Supply flow,-78.0,24986.0,25000.0,False


### Check: cycles with no reference threshold

Cycles are matched to the workbook using the equipment code in the object name (for example `C02C01`). The M01W01 and M01W02 rows in the workbook use short names (`RMT1`, `Milk Reception`, `Raw Milk Tanker`) with no code, and Milk Past-2 is labelled with `C10W02` in the workbook instead of `C01W02`. These cycles are never checked.

In [20]:
# Check: which log objects are not matched, and which workbook rows are never used
no_reference = cycle_results[cycle_results['Status'].eq('NO REFERENCE')]
print(f"{len(no_reference)} of {len(cycle_results)} cycles have no reference threshold "
      f"({len(no_reference) / len(cycle_results):.0%})")
display(no_reference.groupby(['Line', 'Object', 'Object Key']).size().rename('Cycles').reset_index())

unused_reference = thresholds.loc[~thresholds['Object'].isin(logs['Reference Object']), ['Object', 'Object Key']]
print('Workbook rows that no log object matched:')
display(unused_reference)

print('Milk Past-2: log object vs workbook row')
display(pd.DataFrame({
    'Log Object': ['C01W02-Milk Past-2'],
    'Log Object Key': [object_key('C01W02-Milk Past-2')],
    'Workbook Object': [summary.loc[summary['Object'].str.contains('MILK PAST 2'), 'Object'].iloc[0]],
    'Workbook Object Key': [object_key(summary.loc[summary['Object'].str.contains('MILK PAST 2'), 'Object'].iloc[0])],
}))

print('Workbook rows with a missing threshold:')
threshold_view[threshold_view.drop(columns=['Object', 'Object Key']).isna().any(axis=1)]

211 of 699 cycles have no reference threshold (30%)


,Line,Object,Object Key,Cycles
0,M01W01,C01C01-RMT Tank-1,C01C01,21
1,M01W01,C01C02-RMT Tank-2,C01C02,23
2,M01W01,C01C03-RMT Tank-3,C01C03,27
3,M01W01,C01W03-RMT To Reception line,C01W03,21
4,M01W01,R01W01-Milk Reception Line,R01W01-MILK RECEPTION LINE,24
5,M01W02,M01W02_CC01 - Raw Milk Tanker,M01W02_CC01 - RAW MILK TANKER,52
6,M11W02,C01W02-Milk Past-2,C01W02,43


Workbook rows that no log object matched:


,Object,Object Key
12,M11W01_16_CREAM CONC OUTLET-1/2,M11W01_16_CREAM CONC OUTLET-1/2
17,M11W02_05_C03C01-C03W01_OIL TANK,C03C01
22,M11W02_11_C10C01_CREAM TANK-1,C10C01
23,M11W02_12_C10C02_CREAM TANK-2,C10C02
24,M11W02_13_C10W01_RE-CIRCULATION,C10W01
26,M11W02_15_C10W02_MILK PAST 2,C10W02
28,M11W01_17_CPT TO PMT LINE,M11W01_17_CPT TO PMT LINE
29,Milk Reception,MILK RECEPTION
30,RMT1,RMT1
31,RMT2,RMT2


Milk Past-2: log object vs workbook row


,Log Object,Log Object Key,Workbook Object,Workbook Object Key
0,C01W02-Milk Past-2,C01W02,M11W02_15_C10W02_MILK PAST 2,C10W02


Workbook rows with a missing threshold:


,Object,Object Key,Pre Rinse Threshold (min),Caustic Threshold (min),Acid Threshold (min),Final Rinse Total Time (min),Supply Flow Threshold (L/hr),Return Flow Threshold (L/hr)
28,M11W01_17_CPT TO PMT LINE,M11W01_17_CPT TO PMT LINE,NaN,19.35,20.35,18.35,15000,12000


### Check: pre-rinse threshold is the proposed timer, not the timer in use

The pre-rinse threshold is `New Proposed Pre Rinse total time(Min)`. The `Water clear` sheet shows the new timers were applied on 19–21 Aug 2026, but the logs end on 26 Jul 2026. July cycles ran on the old timer, so the cell below compares the alerts against both the proposed and the then-current pre-rinse time.

In [21]:
# Check: log period vs timer change date, and pre-rinse alerts under the current timer
water_clear = pd.read_excel(REFERENCE_FILE, sheet_name='Water clear')
print(f"Log period: {logs['Date Time'].min():%d %b %Y} to {logs['Date Time'].max():%d %b %Y}")
print(f"Timer change dates: {water_clear['Date of Change'].min():%d %b %Y} to {water_clear['Date of Change'].max():%d %b %Y}")

pre_rinse = cycle_results[cycle_results['Pre Rinse'].notna() & cycle_results['Pre Rinse Threshold (min)'].notna()].copy()
pre_rinse['Current Pre Rinse (min)'] = pre_rinse['Reference Object'].map(
    summary.set_index('Object')['Current-Total Pre rinse time(Min)'])
pre_rinse['Breach vs Current'] = pre_rinse['Pre Rinse'] > pre_rinse['Current Pre Rinse (min)'] * (1 + DURATION_TOLERANCE)

print(f"\nPre-rinse cycles checked: {len(pre_rinse)}")
print(f"Pre-rinse alerts vs proposed timer (current notebook): {pre_rinse['Pre Rinse Breach'].sum()}")
print(f"Pre-rinse alerts vs timer in use in July: {pre_rinse['Breach vs Current'].sum()}")
pre_rinse[pre_rinse['Pre Rinse Breach'] != pre_rinse['Breach vs Current']][
    ['Object', 'Cycle Number', 'Pre Rinse', 'Pre Rinse Threshold (min)',
     'Current Pre Rinse (min)', 'Pre Rinse Breach', 'Breach vs Current']].round(2)

Log period: 01 Jul 2026 to 26 Jul 2026
Timer change dates: 19 Aug 2026 to 21 Aug 2026

Pre-rinse cycles checked: 485
Pre-rinse alerts vs proposed timer (current notebook): 64
Pre-rinse alerts vs timer in use in July: 38


,Object,Cycle Number,Pre Rinse,Pre Rinse Threshold (min),Current Pre Rinse (min),Pre Rinse Breach,Breach vs Current
169,C01W01-Milk Past-1,1,15.98,14.50,15.50,True,False
241,C02C02-PMT TANK-2,9,9.00,7.92,8.78,True,False
251,C02C02-PMT TANK-2,19,9.00,7.92,8.78,True,False
281,C02C04-PMT TANK-4,5,8.98,7.92,8.90,True,False
282,C02C04-PMT TANK-4,6,9.00,7.92,8.90,True,False
286,C02C04-PMT TANK-4,10,9.00,7.92,8.90,True,False
287,C02C04-PMT TANK-4,11,9.00,7.92,8.90,True,False
298,C02C04-PMT TANK-4,22,9.00,7.92,8.90,True,False
521,C04W01-HSM Re-Circulation,9,11.02,10.00,11.67,True,False
531,C04W01-HSM Re-Circulation,19,11.02,10.00,11.67,True,False


### Check: smaller issues

1. **Alert count:** "165 alerts" in the earlier summary is the number of cycles in `ALERT` status. The number of individual breaches in `alerts` is higher.
2. **Cell 8 display:** only the last expression in a cell is displayed, so `cycle_counts.head(50)` is never shown (only `unmatched` is). The same applies to `threshold_view.head()` in cell 3.
3. **Duration undercount:** an activity's time is the last sample minus the first sample of that activity, so the minute until the next activity starts is not counted. A one-record activity (often Final Drain) shows 0.0.
4. **Cycle numbering:** `new_cycle.fillna(True)` has no effect because the first `diff()` is `NaT`, and `NaT > 30 min` is `False`, not missing. Cycle numbers start at 0 instead of 1. Cosmetic only.

In [22]:
# 1. Alert count
print(f"Cycles in ALERT status: {cycle_results['Status'].eq('ALERT').sum()}")
print(f"Individual breaches in alerts: {len(alerts)}")

# 2. Tables built in cells 3 and 8 but never displayed
print(f"\ncycle_counts has {len(cycle_counts)} rows and threshold_view has {len(threshold_view)} rows; "
      "neither is shown when cells 3 and 8 run")

# 3. Duration undercount, shown on the first cycle
first = cycle_results.iloc[0]
first_rows = logs[(logs['Object'] == first['Object']) & (logs['Cycle Number'] == first['Cycle Number'])]
blocks = (first_rows.groupby('Activity', sort=False)['Time In Seconds'].agg(['min', 'max', 'size'])
          .rename(columns={'min': 'Start Seconds', 'max': 'Last Sample Seconds', 'size': 'Records'}))
blocks['Next Activity Start'] = blocks['Start Seconds'].shift(-1)
blocks['Notebook (min)'] = (blocks['Last Sample Seconds'] - blocks['Start Seconds']) / 60
blocks['Start To Next Start (min)'] = (blocks['Next Activity Start'] - blocks['Start Seconds']) / 60
print(f"\n{first['Object']} cycle {first['Cycle Number']}:")
display(blocks.round(2))
print(f"Cycles where Final Drain shows 0.0 min: {cycle_results['Final Drain'].eq(0).sum()}")

# 4. Cycle numbering
first_gap = logs.groupby(['Line', 'Object'])['Date Time'].diff().iloc[0]
print(f"\nFirst diff() value: {first_gap}; NaT > 30 min gives {first_gap > pd.Timedelta(minutes=CYCLE_GAP_MINUTES)}")
print(f"Lowest cycle number per object: {sorted(cycle_results.groupby(['Line', 'Object'])['Cycle Number'].min().unique())}")

Cycles in ALERT status: 165
Individual breaches in alerts: 304

cycle_counts has 29 rows and threshold_view has 35 rows; neither is shown when cells 3 and 8 run

C01C01-RMT Tank-1 cycle 0:


,Start Seconds,Last Sample Seconds,Records,Next Activity Start,Notebook (min),Start To Next Start (min)
Activity,,,,,,
Pre Rinse,9,309,6,369.0,5.0,6.0
Caustic,369,1749,24,1809.0,23.0,24.0
Final Rinse,1809,2109,6,2169.0,5.0,6.0
Final Drain,2169,2169,1,NaN,0.0,NaN


Cycles where Final Drain shows 0.0 min: 166

First diff() value: NaT; NaT > 30 min gives False
Lowest cycle number per object: [0]
